# Autocall payoff logic — demo

This notebook demonstrates the `AutocallNote` pricing engine defined in
`autocall_payoff.py`. The path-generation machinery lives in `autocall.ipynb`;
here we regenerate a compact path set the same way and then drive the payoff
engine.

**Pipeline:** simulate correlated paths → attach a date to each path column →
define product `AutocallTerms` → `AutocallNote(...).price()` → inspect the
per-path audit trail.

## 1. Regenerate underlying paths

Identical setup to `autocall.ipynb`: a 2-asset correlated Black–Scholes–Merton
process simulated on a Sobol grid.

In [1]:
import os, sys
import numpy as np
import QuantLib as ql

# Make the module importable from the project root
sys.path.insert(0, os.path.abspath(".."))
from autocall_payoff import AutocallNote, AutocallTerms, BasketMode, RedemptionType

eval_date = ql.Date(19, 5, 2026)
calendar = ql.UnitedStates(ql.UnitedStates.NYSE)
day_count = ql.Actual365Fixed()
ql.Settings.instance().evaluationDate = eval_date

fwd_rate = ql.SimpleQuote(0.03)
zero_curve = ql.FlatForward(eval_date, ql.QuoteHandle(fwd_rate), day_count, ql.Continuous)
zero_curve_handle = ql.YieldTermStructureHandle(zero_curve)

spot_price_1, spot_price_2 = 295.7, 195.7
spot_handle_1 = ql.QuoteHandle(ql.SimpleQuote(spot_price_1))
spot_handle_2 = ql.QuoteHandle(ql.SimpleQuote(spot_price_2))

div_pillars = [ql.Date(21, 5, 2026), ql.Date(19, 6, 2026), ql.Date(18, 9, 2026), ql.Date(17, 12, 2026)]
div_curve_1 = ql.ZeroCurve(div_pillars, [0.0, 0.10358, 0.07464, 0.05098], day_count, calendar)
div_curve_2 = ql.ZeroCurve(div_pillars, [0.0, 0.15358, 0.09464, 0.08098], day_count, calendar)
div_curve_1.enableExtrapolation(); div_curve_2.enableExtrapolation()
div_handle_1 = ql.YieldTermStructureHandle(div_curve_1)
div_handle_2 = ql.YieldTermStructureHandle(div_curve_2)

vol_dates = [ql.Date(19, 8, 2026), ql.Date(19, 11, 2026), ql.Date(19, 2, 2027), ql.Date(19, 5, 2027),
             ql.Date(19, 8, 2027), ql.Date(19, 11, 2027), ql.Date(19, 2, 2028), ql.Date(19, 5, 2028)]
atm_vols_1 = [0.324566, 0.324566, 0.320002, 0.314524, 0.309046, 0.310071, 0.310603, 0.310603]
vol_curve_1 = ql.BlackVarianceCurve(eval_date, vol_dates, atm_vols_1, day_count)
vol_handle = ql.BlackVolTermStructureHandle(vol_curve_1)

bsm_1 = ql.BlackScholesMertonProcess(spot_handle_1, div_handle_1, zero_curve_handle, vol_handle)
bsm_2 = ql.BlackScholesMertonProcess(spot_handle_2, div_handle_2, zero_curve_handle, vol_handle)

rho = 0.65
corr = ql.Matrix(2, 2)
corr[0][0] = corr[1][1] = 1.0
corr[0][1] = corr[1][0] = rho
multi_process = ql.StochasticProcessArray([bsm_1, bsm_2], corr)

In [2]:
sim_dates = [ql.Date(19, 6, 2026), ql.Date(18, 9, 2026), ql.Date(17, 12, 2026),
             ql.Date(19, 2, 2027), ql.Date(19, 5, 2027), ql.Date(19, 8, 2027),
             ql.Date(19, 11, 2027), ql.Date(19, 2, 2028), ql.Date(19, 5, 2028)]
sim_times = [day_count.yearFraction(eval_date, d) for d in sim_dates]
time_grid = ql.TimeGrid(sim_times, len(sim_times))

n_assets = multi_process.size()
n_steps = len(time_grid) - 1
dimensions = n_assets * n_steps

sobol = ql.UniformLowDiscrepancySequenceGenerator(dimensions, 0, ql.SobolRsg.JoeKuoD7)
gaussian = ql.GaussianLowDiscrepancySequenceGenerator(sobol)
path_gen = ql.GaussianSobolMultiPathGenerator(multi_process, time_grid, gaussian, False)

n_paths = 2 ** 12
paths = np.zeros((n_paths, n_assets, n_steps + 1))
for i in range(n_paths):
    multi_path = path_gen.next().value()
    for a in range(n_assets):
        asset_path = multi_path[a]
        paths[i, a] = [asset_path[t] for t in range(n_steps + 1)]

print("paths.shape:", paths.shape)

# Column 0 is the valuation date; columns 1..n are the simulation dates.
path_dates = [eval_date] + sim_dates
schedule = sim_dates                       # one observation per simulation date
initial_fixings = [spot_price_1, spot_price_2]

# Payment dates may differ from observation dates: here each cashflow settles 5
# business days after the observation that fixes it. They are used only for
# discounting (no fixing is read there). Omit payment_dates (or pass
# payment_dates=schedule) to settle on the observation date.
settlement_lag = 5
payment_dates = [calendar.advance(d, settlement_lag, ql.Days) for d in schedule]

# Notional (position size). The per-unit price is unaffected; it scales the
# market value (notional * price) and the *_notional cashflow columns.
notional = 1_000_000.0

paths.shape: (4096, 2, 10)


## 2. Worst-of, 'usual' early redemption

Basket metric = `min` of constituent performances; the note is redeemed when
that worst performer is **at or above** the 100% barrier. Guaranteed coupon of
2% is paid every surviving period; early redemption pays a 4% per-period
snowball coupon.

In [3]:
terms = AutocallTerms(
    autocall_barrier=1.00,
    redemption_type=RedemptionType.USUAL,
    basket_mode=BasketMode.WORST_OF,
    early_redemption_rate=0.04,
    snowball=True,
    guaranteed_coupon=0.02,
    coupon_digital=False,
    coupon_on_redemption=False,
    notional=notional,
)

note = AutocallNote(
    terms=terms,
    initial_fixings=initial_fixings,
    paths=paths,
    path_dates=path_dates,
    schedule=schedule,
    discount_curve=zero_curve_handle,
    payment_dates=payment_dates,
)
price = note.price()
note.summary()

{'price': 0.14218186778359426,
 'stderr': 0.0015450261481973833,
 'notional': 1000000.0,
 'market_value': 142181.86778359427,
 'mv_stderr': 1545.0261481973832,
 'n_paths': 4096,
 'redemption_prob_total': 0.646484375}

In [4]:
note.summary()


{'price': 0.14218186778359426,
 'stderr': 0.0015450261481973833,
 'notional': 1000000.0,
 'market_value': 142181.86778359427,
 'mv_stderr': 1545.0261481973832,
 'n_paths': 4096,
 'redemption_prob_total': 0.646484375}

In [5]:
note.audit

date payment_date    perf_0    perf_1  basket_metric  \
path obs                                                               
0    0    2026-06-19   2026-06-26  0.989388  0.985251       0.985251   
     1    2026-09-18   2026-09-25  0.968167  0.961645       0.961645   
     2    2026-12-17   2026-12-24  0.958738  0.942057       0.942057   
     3    2027-02-19   2027-02-26  0.956382  0.930994       0.930994   
     4    2027-05-19   2027-05-26  0.954089  0.916762       0.916762   
...              ...          ...       ...       ...            ...   
4095 4    2027-05-19   2027-05-26  1.022500  0.613593       0.613593   
     5    2027-08-19   2027-08-26  1.083041  0.742583       0.742583   
     6    2027-11-19   2027-11-29  1.069671  0.749611       0.749611   
     7    2028-02-19   2028-02-28  1.117003  0.834742       0.834742   
     8    2028-05-19   2028-05-26  0.904291  0.721589       0.721589   

          basket_performance  redeemed  coupon  redemption_interest  cashflow  \
path obs                                                                        
0    0              0.985251     False    0.02                  0.0      0.02   
     1              0.961645     False    0.02                  0.0      0.02   
     2              0.942057     False    0.02                  0.0      0.02   
     3              0.930994     False    0.02                  0.0      0.02   
     4              0.916762     False    0.02                  0.0      0.02   
...                      ...       ...     ...                  ...       ...   
4095 4              0.613593     False    0.02                  0.0      0.02   
     5              0.742583     False    0.02                  0.0      0.02   
     6              0.749611     False    0.02                  0.0      0.02   
     7              0.834742     False    0.02                  0.0      0.02   
     8              0.721589     False    0.02                  0.0      0.02   

          discount_factor  discounted_cf  cashflow_notional  \
path obs                                                      
0    0           0.996882       0.019938            20000.0   
     1           0.989453       0.019789            20000.0   
     2           0.982161       0.019643            20000.0   
     3           0.977008       0.019540            20000.0   
     4           0.969887       0.019398            20000.0   
...                   ...            ...                ...   
4095 4           0.969887       0.019398            20000.0   
     5           0.962581       0.019252            20000.0   
     6           0.955094       0.019102            20000.0   
     7           0.947977       0.018960            20000.0   
     8           0.941145       0.018823            20000.0   

          discounted_cf_notional  
path obs                          
0    0              19937.631694  
     1              19789.065424  
     2              19643.220647  
     3              19540.163218  
     4              19397.747112  
...                          ...  
4095 4              19397.747112  
     5              19251.621429  
     6              19101.885894  
     7              18959.547225  
     8              18822.909894  

[36864 rows x 14 columns]

In [6]:
note.expected_cashflows()

,date,payment_date,expected_performance,redemption_prob,ko_coupon_notional,guaranteed_coupon_notional,discount_factor,expected_discounted_cf_notional
obs,,,,,,,,
0,2026-06-19,2026-06-26,0.960356,0.310059,40000.0,20000.0,0.996882,26119.465740
1,2026-09-18,2026-09-25,0.920484,0.136963,80000.0,20000.0,0.989453,21784.398436
2,2026-12-17,2026-12-24,0.898864,0.064941,120000.0,20000.0,0.982161,17240.570856
3,2027-02-19,2027-02-26,0.889183,0.032959,160000.0,20000.0,0.977008,14044.492313
4,2027-05-19,2027-05-26,0.878265,0.028076,200000.0,20000.0,0.969887,13729.020722
5,2027-08-19,2027-08-26,0.869531,0.020752,240000.0,20000.0,0.962581,12615.076151
6,2027-11-19,2027-11-29,0.859191,0.021973,280000.0,20000.0,0.955094,13216.490386
7,2028-02-19,2028-02-28,0.849964,0.017090,320000.0,20000.0,0.947977,12145.959941
8,2028-05-19,2028-05-26,0.842269,0.013672,360000.0,20000.0,0.941145,11286.393237


In [7]:
note.paths


array([[[295.7       , 292.56203017, 286.28708925, ..., 280.55162929,
         279.49322215, 278.52934227],
        [195.7       , 192.81353541, 188.19400174, ..., 173.67759821,
         170.71202064, 167.90069976]],

       [[295.7       , 301.85172453, 255.5053506 , ..., 311.23988616,
         326.63450117, 342.52954779],
        [195.7       , 209.84323035, 194.1349063 , ..., 175.58186003,
         198.70739804, 224.35643055]],

       [[295.7       , 283.55823254, 320.77722552, ..., 252.88923495,
         239.15557281, 226.48730601],
        [195.7       , 177.16587462, 182.43490038, ..., 171.79398895,
         146.6608404 , 125.65115655]],

       ...,

       [[295.7       , 290.86874846, 322.86333601, ..., 328.65073656,
         284.22844219, 285.77220719],
        [195.7       , 193.95638646, 222.11776245, ..., 196.52930304,
         139.74401694, 142.34267822]],

       [[295.7       , 230.67166606, 231.31538581, ..., 250.90573613,
         251.17452444, 220.96766971],
       

### Per-path audit trail

Every `(path, observation)` row records the constituent performances, basket
metric, redemption flag, coupon, interest, total and discounted cashflow.

In [9]:
# First path that actually autocalls, shown end-to-end
redeemed_paths = note.audit.groupby("path")["redeemed"].any()
example_path = redeemed_paths[redeemed_paths].index[0]
note.audit.loc[example_path].round(4)

,date,payment_date,perf_0,perf_1,basket_metric,basket_performance,redeemed,coupon,redemption_interest,cashflow,discount_factor,discounted_cf,cashflow_notional,discounted_cf_notional
obs,,,,,,,,,,,,,,
0,2026-06-19,2026-06-26,1.0208,1.0723,1.0208,1.0208,True,0.0,0.04,0.04,0.9969,0.0399,40000.0,39875.2634
1,2026-09-18,2026-09-25,0.8641,0.9920,0.8641,0.8641,False,0.0,0.00,0.00,0.9895,0.0000,0.0,0.0000
2,2026-12-17,2026-12-24,0.9865,1.0242,0.9865,0.9865,False,0.0,0.00,0.00,0.9822,0.0000,0.0,0.0000
3,2027-02-19,2027-02-26,0.9427,0.9010,0.9010,0.9010,False,0.0,0.00,0.00,0.9770,0.0000,0.0,0.0000
4,2027-05-19,2027-05-26,1.0723,0.9313,0.9313,0.9313,False,0.0,0.00,0.00,0.9699,0.0000,0.0,0.0000
5,2027-08-19,2027-08-26,1.2173,0.9619,0.9619,0.9619,False,0.0,0.00,0.00,0.9626,0.0000,0.0,0.0000
6,2027-11-19,2027-11-29,1.0526,0.8972,0.8972,0.8972,False,0.0,0.00,0.00,0.9551,0.0000,0.0,0.0000
7,2028-02-19,2028-02-28,1.1046,1.0154,1.0154,1.0154,False,0.0,0.00,0.00,0.9480,0.0000,0.0,0.0000
8,2028-05-19,2028-05-26,1.1584,1.1464,1.1464,1.1464,False,0.0,0.00,0.00,0.9411,0.0000,0.0,0.0000


In [10]:
# Expected (mean) cashflow profile across all paths
note.expected_cashflows().round(4)

,date,payment_date,expected_performance,redemption_prob,ko_coupon_notional,guaranteed_coupon_notional,discount_factor,expected_discounted_cf_notional
obs,,,,,,,,
0,2026-06-19,2026-06-26,0.9604,0.3101,40000.0,20000.0,0.9969,26119.4657
1,2026-09-18,2026-09-25,0.9205,0.1370,80000.0,20000.0,0.9895,21784.3984
2,2026-12-17,2026-12-24,0.8989,0.0649,120000.0,20000.0,0.9822,17240.5709
3,2027-02-19,2027-02-26,0.8892,0.0330,160000.0,20000.0,0.9770,14044.4923
4,2027-05-19,2027-05-26,0.8783,0.0281,200000.0,20000.0,0.9699,13729.0207
5,2027-08-19,2027-08-26,0.8695,0.0208,240000.0,20000.0,0.9626,12615.0762
6,2027-11-19,2027-11-29,0.8592,0.0220,280000.0,20000.0,0.9551,13216.4904
7,2028-02-19,2028-02-28,0.8500,0.0171,320000.0,20000.0,0.9480,12145.9599
8,2028-05-19,2028-05-26,0.8423,0.0137,360000.0,20000.0,0.9411,11286.3932


## 3. Memory-on-KO redemption

The note is redeemed once **each** constituent has individually breached the KO
level (= barrier) at least once. For a worst-of basket a breach means the
constituent fixing is at or above the barrier; breaches are memorized for the
remaining life. `basket_metric` here reports how many constituents have breached
so far.

In [11]:
memory_terms = AutocallTerms(
    autocall_barrier=1.00,
    redemption_type=RedemptionType.MEMORY_KO,
    basket_mode=BasketMode.WORST_OF,
    early_redemption_rate=0.04,
    snowball=True,
    guaranteed_coupon=0.02,
)
memory_note = AutocallNote(
    terms=memory_terms,
    initial_fixings=initial_fixings,
    paths=paths,
    path_dates=path_dates,
    schedule=schedule,
    discount_curve=zero_curve_handle,
)
memory_note.price()
memory_note.summary()

{'price': 0.14216758552483244,
 'stderr': 0.0015618253047745422,
 'notional': 1.0,
 'market_value': 0.14216758552483244,
 'mv_stderr': 0.0015618253047745422,
 'n_paths': 4096,
 'redemption_prob_total': 0.677490234375}

## 4. Digital guaranteed coupon + multiperformance basket

A weighted-basket ('multiperformance') trigger with a **digital** guaranteed
coupon: the accumulated `c * k` is paid as a single lump on the termination
date instead of period by period.

In [12]:
multi_terms = AutocallTerms(
    autocall_barrier=1.00,
    redemption_type=RedemptionType.USUAL,
    basket_mode=BasketMode.MULTIPERFORMANCE,
    weights=[0.5, 0.5],
    early_redemption_rate=0.04,
    snowball=False,
    guaranteed_coupon=0.02,
    coupon_digital=True,
    coupon_on_redemption=True,
)
multi_note = AutocallNote(
    terms=multi_terms,
    initial_fixings=initial_fixings,
    paths=paths,
    path_dates=path_dates,
    schedule=schedule,
    discount_curve=zero_curve_handle,
)
multi_note.price()
multi_note.expected_cashflows().round(4)

,date,payment_date,expected_performance,redemption_prob,ko_coupon_notional,guaranteed_coupon_notional,discount_factor,expected_discounted_cf_notional
obs,,,,,,,,
0,2026-06-19,2026-06-19,0.9917,0.4441,0.04,0.02,0.9975,0.0266
1,2026-09-18,2026-09-18,0.9819,0.1521,0.04,0.04,0.9900,0.0120
2,2026-12-17,2026-12-17,0.9793,0.0667,0.04,0.06,0.9827,0.0065
3,2027-02-19,2027-02-19,0.9806,0.0330,0.04,0.08,0.9776,0.0039
4,2027-05-19,2027-05-19,0.9825,0.0271,0.04,0.10,0.9704,0.0037
5,2027-08-19,2027-08-19,0.9845,0.0222,0.04,0.12,0.9631,0.0034
6,2027-11-19,2027-11-19,0.9864,0.0154,0.04,0.14,0.9559,0.0026
7,2028-02-19,2028-02-19,0.9884,0.0156,0.04,0.16,0.9487,0.0030
8,2028-05-19,2028-05-19,0.9902,0.0098,0.04,0.18,0.9417,0.0383
